# GSM8K Math Problems With Difficulty Tiers

This notebook demonstrates the **GSM8K dataset** prepared for a self-check divergence experiment investigating whether **shorter chain-of-thought reduces contradiction in model self-checks**.

The dataset contains math word problems from the well-established GSM8K benchmark (Cobbe et al. 2021), stratified into three balanced difficulty tiers (easy, medium, hard) using quantile-based binning on a composite difficulty score derived from: operation count, numeric complexity, sentence count, token length, and multi-step reasoning indicators.

**What this notebook does:**
1. Loads the curated GSM8K dataset
2. Demonstrates the difficulty scoring pipeline (feature extraction, normalization, tier assignment)
3. Visualizes the difficulty distribution across tiers
4. Shows sample problems from each difficulty level

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install (only if missing)
try:
    import loguru
except ImportError:
    _pip('loguru')

# Core packages — pre-installed on Colab, install locally only (only if missing)
if 'google.colab' not in sys.modules:
    missing = []
    for pkg in ['numpy', 'pandas', 'matplotlib', 'seaborn']:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)
    if missing:
        _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'seaborn==0.13.2')

In [2]:
import json
import re
import random
import math
import sys
from pathlib import Path
from loguru import logger
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Configure logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

random.seed(42)

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-febf9b-longer-reasoning-chains-reduce-self/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os

In [4]:
data = load_data()
print(f"Loaded dataset: {data['metadata']['source']}")
print(f"Total problems in full dataset: {data['metadata']['total_problems']}")
print(f"Tier distribution (full): {data['metadata']['tier_distribution']}")
print(f"Examples in this demo: {len(data['datasets'][0]['examples'])}")

Loaded dataset: openai/gsm8k
Total problems in full dataset: 8792
Tier distribution (full): {'easy': 2930, 'medium': 2930, 'hard': 2932}
Examples in this demo: 17


In [5]:
# === Tunable Parameters ===
# Set to absolute minimum values for a quick demo.
# Original values from data.py: per_tier=17 (51 total mini), preview=(2,2,1)

N_SAMPLES = len(data['datasets'][0]['examples'])  # Use all demo examples (17)
MINI_PER_TIER = 5  # Original: 17 per tier (51 total)
PREVIEW_EASY = 2
PREVIEW_MEDIUM = 2
PREVIEW_HARD = 1

# Difficulty score weights (from original script)
WEIGHT_OPERATION = 0.30
WEIGHT_NUMERIC = 0.25
WEIGHT_SENTENCE = 0.20
WEIGHT_TOKEN = 0.15
WEIGHT_MULTI_STEP = 0.10

print(f"Config: N_SAMPLES={N_SAMPLES}, MINI_PER_TIER={MINI_PER_TIER}")
print(f"Preview counts: easy={PREVIEW_EASY}, medium={PREVIEW_MEDIUM}, hard={PREVIEW_HARD}")

Config: N_SAMPLES=17, MINI_PER_TIER=5
Preview counts: easy=2, medium=2, hard=1


## Answer Extraction

The original script extracts the final numeric answer from GSM8K solution text using the `#### X` format, with a fallback to `<<...>>` patterns.

In [6]:
def extract_answer(answer_text: str) -> str:
    """Extract final numeric answer from GSM8K solution text (#### X format)."""
    match = re.search(r'####\s*(.+?)\s*$', answer_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Fallback: last <<...>> pattern
    matches = re.findall(r'<<([^>>]+)>>', answer_text)
    if matches:
        return matches[-1].strip()
    return ""

# Test on first example
first_ex = data['datasets'][0]['examples'][0]
extracted = extract_answer(first_ex['metadata_solution_steps'])
print(f"Question: {first_ex['input'][:80]}...")
print(f"Extracted answer: {extracted}")
print(f"Ground truth: {first_ex['output']}")
print(f"Match: {extracted == first_ex['output']}")

Question: In his training as a professional athlete, Tyson runs 5000 meters every day. His...
Extracted answer: 180000
Ground truth: 180000
Match: True


## Difficulty Feature Extraction

Compute difficulty features for each problem: operation count, numeric count, sentence count, token count, and multi-step reasoning indicators.

In [7]:
def count_operations(text: str) -> int:
    """Count arithmetic operations in solution text."""
    ops = re.findall(r'[\+\-\*/]', text)
    return len(ops)

def count_numerics(text: str) -> int:
    """Count distinct numeric values in solution text."""
    numbers = re.findall(r'\b\d+\.?\d*\b', text)
    return len(set(numbers))

def count_sentences(text: str) -> int:
    """Count sentences in solution text."""
    sentences = re.split(r'[.!?]+', text)
    return len([s for s in sentences if s.strip()])

def count_tokens(text: str) -> int:
    """Count words in question text."""
    return len(text.split())

def count_multi_step_indicators(text: str) -> int:
    """Count multi-step reasoning indicators."""
    indicators = ['then', 'if', 'total', 'remaining', 'each', 'per', 'next', 'after', 'finally', 'first']
    text_lower = text.lower()
    return sum(1 for ind in indicators if ind in text_lower)

def compute_difficulty_score(question: str, answer: str) -> dict:
    """Compute difficulty score and features for a problem."""
    op_count = count_operations(answer)
    numeric_count = count_numerics(answer)
    sentence_count = count_sentences(answer)
    token_count = count_tokens(question)
    multi_step = count_multi_step_indicators(question + " " + answer)
    return {
        "operation_count": op_count,
        "numeric_count": numeric_count,
        "sentence_count": sentence_count,
        "token_count": token_count,
        "multi_step_indicators": multi_step,
    }

# Compute features for all demo examples
examples = data['datasets'][0]['examples'][:N_SAMPLES]
processed = []
for ex in examples:
    features = compute_difficulty_score(ex['input'], ex['metadata_solution_steps'])
    processed.append({
        'question': ex['input'],
        'answer_text': ex['metadata_solution_steps'],
        'ground_truth': ex['output'],
        'features': features,
        'split': ex['metadata_split'],
        'original_tier': ex['metadata_difficulty_tier'],
        'original_score': ex['metadata_difficulty_score'],
    })

print(f"Computed features for {len(processed)} problems")
print(f"Sample features: {processed[0]['features']}")

Computed features for 17 problems
Sample features: {'operation_count': 9, 'numeric_count': 7, 'sentence_count': 4, 'token_count': 47, 'multi_step_indicators': 3}


## Score Normalization and Tier Assignment

Normalize all feature scores to [0, 1] range, compute weighted difficulty scores, and assign tiers using quantile-based binning.

In [8]:
def normalize_scores(features_list: list) -> list:
    """Normalize all feature scores to [0, 1] range."""
    if not features_list:
        return []
    keys = ["operation_count", "numeric_count", "sentence_count", "token_count", "multi_step_indicators"]
    normalized = []
    for features in features_list:
        norm = {}
        for key in keys:
            vals = [f[key] for f in features_list]
            min_val, max_val = min(vals), max(vals)
            if max_val > min_val:
                norm[key] = (features[key] - min_val) / (max_val - min_val)
            else:
                norm[key] = 0.0
        normalized.append(norm)
    return normalized

def compute_final_score(norm: dict) -> float:
    """Compute weighted difficulty score."""
    score = (
        WEIGHT_OPERATION * norm["operation_count"]
        + WEIGHT_NUMERIC * norm["numeric_count"]
        + WEIGHT_SENTENCE * norm["sentence_count"]
        + WEIGHT_TOKEN * norm["token_count"]
        + WEIGHT_MULTI_STEP * norm["multi_step_indicators"]
    )
    return round(score, 4)

def assign_tiers_quantile(processed: list) -> None:
    """Assign difficulty tiers using quantile-based binning for balanced distribution."""
    sorted_problems = sorted(processed, key=lambda p: p["difficulty_score"])
    n = len(sorted_problems)
    third = n // 3
    for i, p in enumerate(sorted_problems):
        if i < third:
            p["difficulty_tier"] = "easy"
        elif i < 2 * third:
            p["difficulty_tier"] = "medium"
        else:
            p["difficulty_tier"] = "hard"

# Normalize and score
norm_features = normalize_scores([p["features"] for p in processed])
for i, p in enumerate(processed):
    p["norm_features"] = norm_features[i]
    p["difficulty_score"] = compute_final_score(norm_features[i])

assign_tiers_quantile(processed)

# Count tiers
tier_counts = {"easy": 0, "medium": 0, "hard": 0}
for p in processed:
    tier_counts[p["difficulty_tier"]] += 1

print(f"Tier distribution (recomputed on {len(processed)} samples): {tier_counts}")
print(f"Score range: [{min(p['difficulty_score'] for p in processed):.4f}, {max(p['difficulty_score'] for p in processed):.4f}]")

Tier distribution (recomputed on 17 samples): {'easy': 5, 'medium': 5, 'hard': 7}
Score range: [0.0775, 0.5848]


## Mini and Preview Subset Creation

Create balanced mini and preview subsets from the processed data, following the original script's sampling logic.

In [9]:
# Build examples for output
output_examples = []
for idx, p in enumerate(processed):
    example = {
        "input": p["question"],
        "output": p["ground_truth"],
        "metadata_difficulty_tier": p["difficulty_tier"],
        "metadata_difficulty_score": p["difficulty_score"],
        "metadata_solution_steps": p["answer_text"],
        "metadata_operation_count": p["features"]["operation_count"],
        "metadata_numeric_count": p["features"]["numeric_count"],
        "metadata_sentence_count": p["features"]["sentence_count"],
        "metadata_token_count": p["features"]["token_count"],
        "metadata_multi_step_indicators": p["features"]["multi_step_indicators"],
        "metadata_split": p["split"],
        "metadata_row_index": idx,
    }
    output_examples.append(example)

# Create mini subset (balanced tiers)
tier_groups = {"easy": [], "medium": [], "hard": []}
for ex in output_examples:
    tier = ex["metadata_difficulty_tier"]
    tier_groups[tier].append(ex)

mini_examples = []
for tier in ["easy", "medium", "hard"]:
    sampled = random.sample(tier_groups[tier], min(MINI_PER_TIER, len(tier_groups[tier])))
    mini_examples.extend(sampled)
random.shuffle(mini_examples)

print(f"Mini subset: {len(mini_examples)} examples")
for tier in ["easy", "medium", "hard"]:
    count = sum(1 for e in mini_examples if e['metadata_difficulty_tier'] == tier)
    print(f"  {tier}: {count}")

# Create preview subset
preview_examples = []
for tier, count in [("easy", PREVIEW_EASY), ("medium", PREVIEW_MEDIUM), ("hard", PREVIEW_HARD)]:
    sampled = random.sample(tier_groups[tier], min(count, len(tier_groups[tier])))
    preview_examples.extend(sampled)

print(f"\nPreview subset: {len(preview_examples)} examples")

Mini subset: 15 examples
  easy: 5
  medium: 5
  hard: 5

Preview subset: 5 examples


## Results and Visualization

Display key statistics and visualize the difficulty distribution across tiers.

In [10]:
import pandas as pd

# === Summary Table ===
print("=" * 80)
print("GSM8K DATASET SUMMARY")
print("=" * 80)
print(f"Source: {data['metadata']['source']}")
print(f"Full dataset: {data['metadata']['total_problems']} problems")
print(f"Full tier distribution: {data['metadata']['tier_distribution']}")
print(f"Demo samples: {len(processed)} problems")
print(f"Recomputed tier distribution: {tier_counts}")
print(f"Difficulty score range: [{min(p['difficulty_score'] for p in processed):.4f}, {max(p['difficulty_score'] for p in processed):.4f}]")
print(f"Mean difficulty score: {sum(p['difficulty_score'] for p in processed)/len(processed):.4f}")
print()

# Sample problems by tier
print("SAMPLE PROBLEMS BY TIER")
print("-" * 80)
for tier in ["easy", "medium", "hard"]:
    tier_samples = [p for p in processed if p["difficulty_tier"] == tier][:2]
    for p in tier_samples:
        q = p['question'][:100] + "..." if len(p['question']) > 100 else p['question']
        print(f"\n[{tier.upper()}] (score: {p['difficulty_score']:.4f})")
        print(f"  Q: {q}")
        print(f"  A: {p['ground_truth']}")
        print(f"  Features: ops={p['features']['operation_count']}, nums={p['features']['numeric_count']}, sent={p['features']['sentence_count']}, tokens={p['features']['token_count']}, multi={p['features']['multi_step_indicators']}")

# === Visualization ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Tier distribution bar chart
tiers = ["easy", "medium", "hard"]
counts = [tier_counts[t] for t in tiers]
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].bar(tiers, counts, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Difficulty Tier Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Problems')
axes[0].set_xlabel('Difficulty Tier')
for i, (t, c) in enumerate(zip(tiers, counts)):
    axes[0].text(i, c + 0.1, str(c), ha='center', fontweight='bold')

# Plot 2: Difficulty score distribution by tier
tier_scores = {t: [p['difficulty_score'] for p in processed if p['difficulty_tier'] == t] for t in tiers}
sns.boxplot(data=[tier_scores[t] for t in tiers], ax=axes[1], palette=colors)
axes[1].set_xticklabels(tiers)
axes[1].set_title('Difficulty Score by Tier', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Difficulty Score')
axes[1].set_xlabel('Difficulty Tier')

# Plot 3: Feature radar-style comparison (mean features by tier)
feature_names = ["operations", "numerics", "sentences", "tokens", "multi-step"]
feature_keys = ["operation_count", "numeric_count", "sentence_count", "token_count", "multi_step_indicators"]
tier_means = {}
for t in tiers:
    tier_data = [p['features'] for p in processed if p['difficulty_tier'] == t]
    tier_means[t] = [sum(d[k] for d in tier_data) / len(tier_data) if tier_data else 0 for k in feature_keys]

x_pos = range(len(feature_names))
width = 0.25
for i, t in enumerate(tiers):
    axes[2].bar([x + i*width for x in x_pos], tier_means[t], width, label=t, color=colors[i], alpha=0.8)
axes[2].set_xticks([x + width for x in x_pos])
axes[2].set_xticklabels(feature_names, rotation=30, ha='right')
axes[2].set_title('Mean Feature Values by Tier', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Average Count')
axes[2].legend(title='Tier')

plt.tight_layout()
plt.savefig('difficulty_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nVisualization saved to 'difficulty_visualization.png'")

GSM8K DATASET SUMMARY
Source: openai/gsm8k
Full dataset: 8792 problems
Full tier distribution: {'easy': 2930, 'medium': 2930, 'hard': 2932}
Demo samples: 17 problems
Recomputed tier distribution: {'easy': 5, 'medium': 5, 'hard': 7}
Difficulty score range: [0.0775, 0.5848]
Mean difficulty score: 0.3219

SAMPLE PROBLEMS BY TIER
--------------------------------------------------------------------------------

[EASY] (score: 0.1729)
  Q: Annika brought $50 to the town fair. She spent half of it on food and snacks, and an additional $10 ...
  A: 15
  Features: ops=6, nums=5, sent=3, tokens=28, multi=1

[EASY] (score: 0.0775)
  Q: Mr. Sam shared a certain amount of money between his two sons, Ken and Tony. If Ken got $1750, and T...
  A: 5250
  Features: ops=4, nums=4, sent=1, tokens=33, multi=2

[MEDIUM] (score: 0.2772)
  Q: To have the car for the weekend, Wilson's report card needs to show that he received 80 or higher in...
  A: 80
  Features: ops=6, nums=8, sent=1, tokens=52, multi=0

[

/tmp/ipykernel_43665/3517006912.py:45: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[1].set_xticklabels(tiers)



Visualization saved to 'difficulty_visualization.png'
